# Auditoría de Calidad de Datos - ALOHAS

Este notebook contiene la auditoría inicial de calidad de datos para el caso de estudio de **Analytics Engineer** de ALOHAS. 
El objetivo es identificar anomalías, duplicados y problemas de integridad antes de proceder con el modelado en dbt.

## 1. Configuración del Entorno y Conexión a DuckDB

In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb://
%sql ROLLBACK

,Success


## 2. Inspección Rápida de las Fuentes de Datos

### Catálogo de Productos (`dim_product`) (productos "p")

In [2]:
%%sql
SELECT *
FROM 'data/dim_product.csv'
LIMIT 3;

,sku,name,category,base_price,cost
0,SKU-07957,PaleGoldenRod Strong Accessorie 7957,Accessories,80,32.24
1,SKU-04551,BurlyWood Fine Accessorie 4551,Accessories,80,34.25
2,SKU-07725,Peru Tall Accessorie 7725,Accessories,80,31.97


### Envíos Realizados (`fct_shipment`) (envíos "e")

In [3]:
%%sql
SELECT *
FROM 'data/fct_shipment.csv'
LIMIT 3;

,shipment_id,shipping_method,shipping_cost,country
0,SHP-0000005,pickup,0.0,DE
1,SHP-0000014,pickup,0.0,DE
2,SHP-0000111,pickup,0.0,DE


### Líneas de Orden de Venta (`fct_sale_order_line`) (ventas "v")

In [4]:
%%sql
SELECT *
FROM 'data/fct_sale_order_line.csv'
LIMIT 3;

,channel,sku,shipment_id,quantity_sold,quantity_returned,gross_sale,taxes,net_sales,created_at
0,wholesale,SKU-04081,SHP-0004598,1,0,400,0.0,400.0,2026-03-10 23:55:12
1,wholesale,SKU-01807,SHP-0004913,2,0,240,0.0,240.0,2026-03-10 17:20:58
2,online,SKU-04284,SHP-0021795,2,0,800,168.0,632.0,2026-03-10 03:12:07


## 3. Pruebas de Unicidad (Claves Primarias)

Validamos que las claves primarias (`sku` en productos y `shipment_id` en envíos) sean únicas.

In [5]:
%%sql
SELECT COUNT(*) - COUNT(DISTINCT sku) AS total_skus_duplicados
FROM 'data/dim_product.csv';

,total_skus_duplicados
0,0


In [6]:
%%sql
SELECT COUNT(*) - COUNT(DISTINCT shipment_id) AS total_envios_duplicados
FROM 'data/fct_shipment.csv';

,total_envios_duplicados
0,0


## 4. Pruebas de Integridad

Verificamos si existen ventas huérfanas que no tengan correspondencia en el catálogo de productos o en el registro de envíos.

In [7]:
%%sql
SELECT COUNT(*) AS total_ventas_sin_producto
FROM 'data/fct_sale_order_line.csv' v
LEFT JOIN 'data/dim_product.csv' p ON v.sku = p.sku
WHERE p.sku IS NULL;

,total_ventas_sin_producto
0,750


In [8]:
%%sql
SELECT COUNT(*) AS total_ventas_sin_envio
FROM 'data/fct_sale_order_line.csv' v
LEFT JOIN 'data/fct_shipment.csv' e ON v.shipment_id = e.shipment_id
WHERE e.shipment_id IS NULL;

,total_ventas_sin_envio
0,500


### ¿Vienen de algún canal en específico las ventas sin numero de envío o producto?

In [9]:
%%sql
SELECT 
    v.channel AS canal,
    COUNT(*) AS total_lineas_huerfanas,
    COUNT(DISTINCT v.shipment_id) AS envios_unicos_huerfanos,
    SUM(CASE WHEN v.shipment_id IS NULL THEN 1 ELSE 0 END) AS envios_nulos
FROM 'data/fct_sale_order_line.csv' v
LEFT JOIN 'data/fct_shipment.csv' e ON v.shipment_id = e.shipment_id
WHERE e.shipment_id IS NULL
GROUP BY v.channel;

,canal,total_lineas_huerfanas,envios_unicos_huerfanos,envios_nulos
0,wholesale,74,74,0.0
1,retail,99,99,0.0
2,online,304,304,0.0
3,marketplace,23,23,0.0


In [10]:
%%sql
SELECT 
    v.channel AS canal,
    v.sku LIKE 'SKU-%' AS formato_sku_estandar,
    COUNT(*) AS total_lineas_huerfanas,
    COUNT(DISTINCT v.sku) AS skus_unicos_huerfanos,
    SUM(v.gross_sale) AS venta_bruta_afectada
FROM 'data/fct_sale_order_line.csv' v
LEFT JOIN 'data/dim_product.csv' p ON v.sku = p.sku
WHERE p.sku IS NULL
GROUP BY v.channel, v.sku LIKE 'SKU-%';

,canal,formato_sku_estandar,total_lineas_huerfanas,skus_unicos_huerfanos,venta_bruta_afectada
0,marketplace,True,37,37,9990.0
1,retail,True,166,165,43020.0
2,online,True,435,426,119470.0
3,wholesale,True,112,111,30760.0


## 5. Pruebas de Consistencia Matemática e Impuestos

Verificamos que las relaciones de cálculo financiero cuadren perfectamente (`venta_neta = venta_bruta - impuestos`) y que se cumplan las reglas de negocio (como impuestos en cero para ventas mayoristas / `wholesale`).

In [11]:
%%sql
SELECT 
    -- 1. Casos donde la ecuación no cuadra (0.01 centavos)
    SUM(CASE WHEN ABS(net_sales - (gross_sale - taxes)) > 0.01 THEN 1 ELSE 0 END) AS errores_calculo_venta_neta,
    
    -- 2. Ventas mayoristas con impuestos superiores a cero
    SUM(CASE WHEN channel = 'wholesale' AND taxes > 0 THEN 1 ELSE 0 END) AS ventas_wholesale_con_impuestos
FROM 'data/fct_sale_order_line.csv';

,errores_calculo_venta_neta,ventas_wholesale_con_impuestos
0,0.0,0.0


Verificamos que la columna gross_sale esta correctamente calculada.

In [12]:
%%sql
SELECT 
    v.gross_sale,
    v.quantity_sold * p.base_price AS calculated_sale,
    v.gross_sale - (v.quantity_sold * p.base_price) AS difference
FROM 'data/fct_sale_order_line.csv' v
JOIN 'data/dim_product.csv' p
ON v.sku = p.sku
WHERE difference != 0;

,gross_sale,calculated_sale,difference


## 6. Pruebas de Límites Físicos y Negativos

Revisamos que no existan valores negativos imposibles y que las devoluciones estén correctamente limitadas por la cantidad vendida.

In [13]:
%%sql
SELECT
    -- Cantidades negativas o en cero en ventas
    SUM(CASE WHEN quantity_sold <= 0 THEN 1 ELSE 0 END) AS total_cantidades_cero_o_negativas,
    SUM(CASE WHEN gross_sale < 0 THEN 1 ELSE 0 END) AS total_ventas_brutas_negativas,
    
    -- Devoluciones que exceden a la venta física
    SUM(CASE WHEN quantity_returned > quantity_sold THEN 1 ELSE 0 END) AS devoluciones_imposibles,
    
    -- Precios o costos inválidos en la lista maestra de productos
    (SELECT COUNT(*) FROM 'data/dim_product.csv' WHERE base_price <= 0 OR cost <= 0) AS total_productos_precio_costo_invalido
FROM 'data/fct_sale_order_line.csv';

,total_cantidades_cero_o_negativas,total_ventas_brutas_negativas,devoluciones_imposibles,total_productos_precio_costo_invalido
0,0.0,0.0,0.0,0


## 7. Conclusiones de la Auditoría

* **Claves Primarias:** 100% limpias. Sin duplicados en `sku` ni en `shipment_id`.
* **Integridad Referencial (Errores detectados):**
  * **750 ventas sin producto** en catálogo (Venta afectada: **203,240 EUR**).
  * **500 ventas sin envío** en el registro.
  * *Decisión dbt:* Excluir estas filas en staging/marts mediante `INNER JOIN` porque la falta de costos de producto y envío imposibilita calcular el Margen de Contribución.
* **Consistencia Financiera:** Perfecta. `gross_sale` coincide con precio catálogo. Impuestos mayoristas son 0. `net_sales` cuadra al centavo.
* **Límites Físicos:** Sin anomalías. Cero negativos y no hay devoluciones mayores a las ventas.